# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union (docs/SCHEMA_EXTENSION.md). `store.extend` extends the schemas a store holds;
`Builders.extend` extends the framework, the meta-schemas every store holds, once per process. Anything but added
members is refused. The cases that extend the framework come last: a notebook's cases share its process, and so the
framework's extensions.

In [ ]:
import { Builders as B, Plain, Validators } from "@mbse/schemas/Framework";
import { assert, raises, same } from "./support.js";

const [Text, Int] = [B.Meta.OfStr, B.Meta.OfInt];
const store: any = new B.Store();
const Knows = store.Schemas.Relation((r: any) => (
  r
    .name("crm.Knows")
    .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
    .uniques([]) // a set
));
const Person = store.Schemas.Object((o: any) => (
  o
    .name("crm.Person")
    .ref(true)
    .properties([
      (t: any) => t.name("name").type([Text]),
      (t: any) => t.name("tags").type([store.Schemas.Container((c: any) => c.index([Int]).element([Text]))]),
    ])
    .adjacencies([(t: any) => t.name("knows").relation([Knows]).me("a")])
));
const Customer = store.Schemas.Object((o: any) => (
  o
    .name("crm.Customer")
    .ref(true)
    .refines([Person])
    .properties([(t: any) => t.name("account").type([Text])])
));
store.Schemas.Object((o: any) => ( // a refinement of a refinement
  o
    .name("crm.Vip")
    .ref(true)
    .refines([Customer])
    .properties([(t: any) => t.name("tier").type([Int])])
    .adjacencies([(t: any) => t.name("fans").relation([Knows]).me("b")])
));
const Id = store.Schemas.Union((u: any) => u.name("crm.Id").branches([(t: any) => t.name("n").type([Int])]));
const EXTENDS = "an extension only adds members to";

## EXT-01 · `store.extend` adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
{
  const ada = store.crm.Person((p: any) => p.name("Ada")); // built before the extension
  store.extend((m: any) => {
    m.Schemas.Object(Person, (o: any) => o.properties([(t: any) => t.name("age").type([Int])]));
    m.Schemas.Object(Person, (o: any) => o.adjacencies([(t: any) => t.relation([Knows]).me("b")])); // anonymous
    m.Schemas.Relation(Knows, (r: any) => r.properties([(t: any) => t.name("since").type([Int])]));
    m.Schemas.Union(Id, (u: any) => u.branches([(t: any) => t.name("s").type([Text])]));
    m.Schemas.Object((o: any) => o.name("crm.Note").properties([(t: any) => t.name("text").type([Text])])); // built: its own
  });
  assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.anonymous.length === 1);
  assert(same([...Customer.properties.keys()], ["name", "tags", "age", "account"])); // inherited
  assert(same([...Knows.entry.properties.keys()], ["since"]) && same(Id.branches.map((b: any) => b.name), ["n", "s"]));
  assert(ada.age.get() === null && store.crm.Person(ada, (p: any) => p.age(36n)).age.get() === 36n); // absent until written
  assert(store.crm.Customer((c: any) => c.name("Bo").age(7n).account("A")).age.get() === 7n);
  assert(store.registered("crm.Note").validate().length === 0 && store.extend((m: any) => m) === store);
}

## EXT-02 · The extension check refuses anything but added members, saying what the extension would change

In [ ]:
for (const [extension, says] of [
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.description("A person")), "'crm.Person': it changes 'description'"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.ref(false)), "'crm.Person': it changes 'ref'"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.properties((adv: any) => adv.remove((e: any) => e.name("name")))),
    "'crm.Person': it removes or replaces what 'properties' holds"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.properties([(t: any) => t.name("name").type([Int])])), // retyped
    "'crm.Person': it removes or replaces what 'properties' holds"],
  [(m: any) => m.Schemas.Relation(Knows, (r: any) => r.links([(t: any) => t.name("c")])),
    "'crm.Knows': it adds to 'links', which an extension cannot"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.properties([(t: any) => t.name("account").type([Text])])),
    "'crm.Person': it adds 'account', which 'crm.Customer' declares"],
  [(m: any) => m.Schemas.Object(Customer, (o: any) => o.properties([(t: any) => t.name("name").type([Text])])),
    "'crm.Customer': it adds 'name', which it inherits"],
  [(m: any) => m.Schemas.Object(Customer, (o: any) => o.adjacencies([(t: any) => t.name("name").relation([Knows]).me("a")])),
    "'crm.Customer': it adds 'name', which it inherits"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.adjacencies([(t: any) => t.name("name").relation([Knows]).me("a")])),
    "'crm.Person': it adds 'name', which it has"],
  [(m: any) => m.Schemas.Object(Customer, (o: any) => o.adjacencies([(t: any) => t.name("knows").relation([Knows]).me("a")])),
    "'crm.Customer': it adds 'knows', which it inherits"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.properties([(t: any) => t.name("tier").type([Int])])),
    "'crm.Person': it adds 'tier', which 'crm.Vip' declares"],
  [(m: any) => m.Schemas.Object(Person, (o: any) => o.properties([(t: any) => t.name("fans").type([Int])])),
    "'crm.Person': it adds 'fans', which 'crm.Vip' declares"],
  [(m: any) => m.Schemas.Container(Person.properties.get("tags").type, (c: any) => c.layout("sparse")), // unnamed: its identity
    "'crm.Person.tags#type': it changes 'layout'"],
] as const) {
  raises(TypeError, () => store.extend(extension), `${EXTENDS} ${says}`);
}
assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.ref && Person.description === null); // refused: unwritten
store.Schemas.Object(Person, (o: any) => o.description("A person")); // outside an extension, a store's own schema changes
assert(Person.description === "A person");

## EXT-03 · A framework schema, which every store holds, changes only through `Builders.extend`

In [ ]:
for (const change of [
  () => store.Schemas.Object(B.Meta.Scalar, (o: any) => o.description("x")),
  () => store.extend((m: any) => m.Schemas.Object(B.Meta.Scalar, (o: any) => o.properties([(t: any) => t.name("unit").type([Text])]))),
]) {
  raises(TypeError, change, "'Schemas.Scalar' is a framework schema, which every store holds: extend it with Builders.extend");
}
assert(!B.Meta.Scalar.properties.has("unit"));
raises(TypeError, () => B.extend("", (m: any) => m), "an extension is named by non-empty text, got ''");

## EXT-04 · `Builders.extend` extends the framework once per process: every store holds what it built, and a meta-schema's new members are its schemas' builders'

In [ ]:
const early: any = new B.Store(); // made before the extension

function domains(m: any): void {
  const Domain = m.Schemas.Object((o: any) => (
    o
      .name("Expressions.Framework.Domain")
      .ref(true)
      .properties([(t: any) => t.name("label").type([Text])])
  ));
  const HasDomain = m.Schemas.Relation((r: any) => (
    r
      .name("Expressions.Framework.HasDomain")
      .links([(t: any) => t.name("schema"), (t: any) => t.name("domain")])
      .uniques("schema")
  ));
  m.Schemas.Object(Domain, (o: any) => o.adjacencies([(t: any) => t.name("schemas").relation([HasDomain]).me("domain")]));
  m.Schemas.Object(B.OfScalar.Schema, (o: any) => (
    o
      .properties([(t: any) => t.name("unit").type([Text])])
      .adjacencies([(t: any) => t.name("domain").relation([HasDomain]).me("schema")])
  ));
}

B.extend("test.domains", domains);
B.extend("test.domains", domains); // applied: nothing again
const late: any = new B.Store();
for (const held of [early, late]) {
  const Domain = held.registered("Expressions.Framework.Domain");
  assert(held.framework(Domain) && held.framework(held.registered("Expressions.Framework.HasDomain")));
}
assert(B.OfScalar.Schema.adjacencies.has("domain") && B.OfScalar.Schema.properties.has("unit"));
const Meters = late.Schemas.Scalar((s: any) => s.name("Meters").token("float").unit("m"));
assert(new B.OfScalar.Builder(Meters, false, late).unit.get() === "m");
const Feet = new B.OfScalar.Builder(Meters, true, late).name("Feet").clone();
assert(new B.OfScalar.Builder(Feet, false, late).unit.get() === "m"); // a clone keeps it
raises(TypeError, () => B.extend("test.trio", (m: any) => m.Schemas.Object(B.OfScalar.Schema, (o: any) => o.adjacencies([
  (t: any) => t.name("trio").relation([m.Schemas.Relation((r: any) => r.name("T3").links([
    (u: any) => u.name("a"), (u: any) => u.name("b"), (u: any) => u.name("c")]))]).me("a")]))),
"an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'");

## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot

In [ ]:
{
  const s: any = new B.Store();
  const int8 = s.Expressions.Framework.Domain((d: any) => d.label("int8"));
  const Byte = s.Schemas.Scalar((x: any) => x.name("Byte").token("int").domain([int8])); // from the schema's end
  assert(same([...int8.schemas].map((e: any) => e.schema === Byte), [true])); // seen from the data's end
  const int16 = s.Expressions.Framework.Domain((d: any) => d.label("int16").schemas((a: any) => a.insert((e: any) => e.schema(Byte))));
  assert(same([...new B.OfScalar.Builder(Byte, false, s).domain].map((e: any) => e.domain.label.get()), ["int8", "int16"]));
  assert(Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8).length === 0);
  const text = (value: unknown) => JSON.stringify(value, (_k, v) => (v instanceof Map ? Object.fromEntries(v) : v));
  const log = Plain.ToPlain(s).Snapshot();
  assert(log.filter((mutation: any) => mutation.has("insert") && text(mutation).includes("HasDomain")).length === 2); // each once
  const back: any = new B.Store();
  Plain.FromPlain(back).Snapshot(log);
  assert([...new B.OfScalar.Builder(back.registered("Byte"), false, back).domain].length === 2);
  assert(text(Plain.ToPlain(back).Snapshot()) === text(log));
  s.Schemas.Scalar(Byte, (x: any) => x.domain((a: any) => a.remove((e: any) => e.domain(int8))));
  assert([...int8.schemas].length === 0 && [...int16.schemas].length === 1);
  raises(TypeError, () => s.Expressions.Framework.Domain((d: any) => d.schemas((a: any) => a.insert((e: any) => e.schema(
    (new B.Store() as any).Schemas.Scalar((x: any) => x))))), "proxies can only link proxy instances, or schema objects their store holds");
}